# Qwen3-14B + LoRA on AWS Inferentia2

This notebook demonstrates running **Qwen3-14B** (text-only, 14.8B parameters) with **LoRA adapters** on AWS Inferentia2 using NxD Inference and vLLM.

## What this notebook does

1. Downloads Qwen3-14B base model and two public LoRA adapters
2. Compiles the model with LoRA support at tp=4 on inf2.24xlarge
3. Runs inference using different LoRA adapters
4. Benchmarks throughput for each adapter
5. Uploads artifacts to HuggingFace for reuse on fresh instances

## Requirements

- **Instance**: inf2.24xlarge (12 NeuronCores, 192 GB HBM, 16 GB/core)
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260227
- **SDK**: Neuron SDK 2.28, NxDI 0.8.x, vLLM-neuron 0.4.1
- **TP degree**: 4 (3.7 GB/core for model weights, 8.6 GB headroom)
- **Precision**: BF16 (FP8 not available on inf2)

## LoRA Adapters

| Adapter | HuggingFace Repo | Rank | Alpha | Size |
|---------|-----------------|------|-------|------|
| Adapter A | `nicoboss/Qwen3-14B-Uncensored-Lora` | 32 | 16 | 514 MB |
| Adapter B | `Wuhall/Qwen3-14B-LoRA` | 32 | 32 | 514 MB |

Both adapters target: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj

## Known Issue

**vLLM-neuron 0.4.1** requires every request to include a `lora_request` when `enable_lora=True`.
Sending a request without one (base model inference) crashes with
`AttributeError: 'NoneType' object has no attribute 'lora_name'`.
An internal ticket has been filed and this will be resolved in a future vLLM-neuron release.
Until then, always pass a `lora_request` with every generate call.

## 0. Environment Setup

In [ ]:
# Activate the pre-installed Neuron venv before running this notebook:
# source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
# jupyter notebook --no-browser --port=8888

import os
import sys
import time
import json

# Must be set before any vLLM/NxDI imports
os.environ["VLLM_NEURON_FRAMEWORK"] = "neuronx-distributed-inference"
os.environ["VLLM_USE_V1"] = "1"

# Configuration
BASE_MODEL_ID = "Qwen/Qwen3-14B"
BASE_MODEL_DIR = os.path.expanduser("~/Qwen3-14B")
LORA_DIR = os.path.expanduser("~/lora_adapters/qwen3-14b")
ARTIFACT_DIR = os.path.expanduser("~/neuron-artifacts/qwen3-14b-tp4-lora")
HF_REPO = "jburtoft/Qwen3-14B-neuron-inf2-tp4-lora"
HF_TOKEN = os.getenv("HF_TOKEN")  # Set your HF token as an env var

# LoRA adapter repos
LORA_ADAPTERS = {
    "adapter_a": "nicoboss/Qwen3-14B-Uncensored-Lora",
    "adapter_b": "Wuhall/Qwen3-14B-LoRA",
}

# TP=4 on inf2.24xlarge (12 cores total, 4 used for TP, remainder for DP if desired)
TP_DEGREE = 4
MAX_MODEL_LEN = 4096
BATCH_SIZE = 1

print(f"Base model: {BASE_MODEL_ID}")
print(f"TP degree: {TP_DEGREE}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"LoRA adapters: {list(LORA_ADAPTERS.keys())}")

Base model: Qwen/Qwen3-14B
TP degree: 4
Artifacts: /home/ubuntu/neuron-artifacts/qwen3-14b-tp4-lora
LoRA adapters: ['adapter_a', 'adapter_b']


In [2]:
# Install dependencies
!pip install -q huggingface_hub safetensors

## 1. Download Model and LoRA Adapters

In [3]:
from huggingface_hub import snapshot_download, hf_hub_download

# Download base model (~30 GB)
if not os.path.exists(os.path.join(BASE_MODEL_DIR, "config.json")):
    print(f"Downloading {BASE_MODEL_ID}...")
    snapshot_download(
        repo_id=BASE_MODEL_ID,
        local_dir=BASE_MODEL_DIR,
        ignore_patterns=["*.gguf", "*.md", "original/*"],
        token=HF_TOKEN,
    )
    print(f"Downloaded to {BASE_MODEL_DIR}")
else:
    print(f"Base model already exists at {BASE_MODEL_DIR}")

Base model already exists at /home/ubuntu/Qwen3-14B


In [4]:
# Verify model config
with open(os.path.join(BASE_MODEL_DIR, "config.json")) as f:
    config = json.load(f)

print(f"Model type: {config['model_type']}")
print(f"Hidden size: {config['hidden_size']}")
print(f"Intermediate size: {config['intermediate_size']}")
print(f"Num layers: {config['num_hidden_layers']}")
print(f"Num attention heads: {config['num_attention_heads']}")
print(f"Num KV heads: {config['num_key_value_heads']}")
print(f"Vocab size: {config['vocab_size']}")
print(f"tie_word_embeddings: {config.get('tie_word_embeddings', 'not set')}")

# Qwen3-14B has tie_word_embeddings=false, so no lm_head fix needed
assert config.get("tie_word_embeddings", True) == False, \
    "Expected tie_word_embeddings=False for Qwen3-14B"

Model type: qwen3
Hidden size: 5120
Intermediate size: 17408
Num layers: 40
Num attention heads: 40
Num KV heads: 8
Vocab size: 151936
tie_word_embeddings: False


In [5]:
# Download LoRA adapters
os.makedirs(LORA_DIR, exist_ok=True)

for adapter_id, repo_id in LORA_ADAPTERS.items():
    adapter_path = os.path.join(LORA_DIR, adapter_id)
    if os.path.exists(os.path.join(adapter_path, "adapter_config.json")):
        print(f"  {adapter_id} already downloaded")
        continue
    
    print(f"  Downloading {repo_id} -> {adapter_path}")
    os.makedirs(adapter_path, exist_ok=True)
    
    hf_hub_download(
        repo_id=repo_id,
        filename="adapter_config.json",
        local_dir=adapter_path,
        token=HF_TOKEN,
    )
    hf_hub_download(
        repo_id=repo_id,
        filename="adapter_model.safetensors",
        local_dir=adapter_path,
        token=HF_TOKEN,
    )
    print(f"  Downloaded {adapter_id}")

# Verify adapter configs
for adapter_id in LORA_ADAPTERS:
    adapter_path = os.path.join(LORA_DIR, adapter_id)
    with open(os.path.join(adapter_path, "adapter_config.json")) as f:
        acfg = json.load(f)
    print(f"\n{adapter_id}:")
    print(f"  rank: {acfg.get('r', 'N/A')}")
    print(f"  alpha: {acfg.get('lora_alpha', 'N/A')}")
    print(f"  target_modules: {acfg.get('target_modules', 'N/A')}")
    safetensors_path = os.path.join(adapter_path, "adapter_model.safetensors")
    size_mb = os.path.getsize(safetensors_path) / 1024 / 1024
    print(f"  size: {size_mb:.1f} MB")

  adapter_a already downloaded
  adapter_b already downloaded

adapter_a:
  rank: 32
  alpha: 16
  target_modules: ['gate_proj', 'down_proj', 'o_proj', 'k_proj', 'up_proj', 'v_proj', 'q_proj']
  size: 490.1 MB

adapter_b:
  rank: 32
  alpha: 32
  target_modules: ['o_proj', 'v_proj', 'k_proj', 'up_proj', 'q_proj', 'gate_proj', 'down_proj']
  size: 490.1 MB


## 2. Create LoRA JSON Config

NxDI requires a JSON file specifying the LoRA adapters. The format is:
```json
{
    "lora-ckpt-dir": "/path/to/lora_adapters",
    "lora-ckpt-paths": {
        "adapter_id": "relative/path/to/adapter"
    }
}
```

Adapters listed in `lora-ckpt-paths` are loaded into HBM (device memory) at startup.
Adapters in `lora-ckpt-paths-cpu` are cached in CPU memory for dynamic swapping.

In [6]:
# Create LoRA config JSON
lora_config = {
    "lora-ckpt-dir": LORA_DIR,
    "lora-ckpt-paths": {
        adapter_id: adapter_id  # relative path under LORA_DIR
        for adapter_id in LORA_ADAPTERS
    },
    "lora-ckpt-paths-cpu": {}  # no CPU-cached adapters for this demo
}

lora_json_path = os.path.join(LORA_DIR, "adapters.json")
with open(lora_json_path, "w") as f:
    json.dump(lora_config, f, indent=2)

print(f"LoRA config written to: {lora_json_path}")
print(json.dumps(lora_config, indent=2))

LoRA config written to: /home/ubuntu/lora_adapters/qwen3-14b/adapters.json
{
  "lora-ckpt-dir": "/home/ubuntu/lora_adapters/qwen3-14b",
  "lora-ckpt-paths": {
    "adapter_a": "adapter_a",
    "adapter_b": "adapter_b"
  },
  "lora-ckpt-paths-cpu": {}
}


## 3. Compile Model with LoRA

### NeuronConfig for Qwen3-14B on inf2.24xlarge

- **tp=4**: 14.8 GB model / 4 cores = 3.7 GB/core, with 8.6 GB headroom per 16 GB core
- **All ISA kernels OFF**: Required on inf2 (NCC_IBVF032 issue)
- **Qwen3 is natively supported**: `NeuronQwen3ForCausalLM` registered as `qwen3` model type

### TP Divisibility Check (14B)

| Dimension | Value | /TP=4 | Divides? |
|-----------|-------|-------|----------|
| intermediate_size | 17408 | 4352 | Yes |
| num_attention_heads | 40 | 10 | Yes |
| num_key_value_heads | 8 | 2 | Yes |
| hidden_size | 5120 | 1280 | Yes |

In [7]:
# Neuron config for Qwen3-14B at tp=4, inf2, all kernels off
text_neuron_config = {
    "batch_size": BATCH_SIZE,
    "ctx_batch_size": BATCH_SIZE,
    "tkg_batch_size": BATCH_SIZE,
    "seq_len": MAX_MODEL_LEN,
    "max_context_length": MAX_MODEL_LEN,
    "torch_dtype": "bfloat16",
    "tp_degree": TP_DEGREE,
    "world_size": TP_DEGREE,
    "enable_bucketing": True,
    "context_encoding_buckets": [512, 1024, 2048, 4096],
    "token_generation_buckets": [512, 1024, 2048, 4096],
    "fused_qkv": True,
    "qkv_kernel_enabled": False,   # ALL kernels OFF on inf2
    "mlp_kernel_enabled": False,
    "attn_kernel_enabled": False,
    "logical_neuron_cores": 1,     # inf2 LNC=1
    "cc_pipeline_tiling_factor": 1,
    "rpl_reduce_dtype": "bfloat16",
    "attention_dtype": "bfloat16",
    "cast_type": "as-declared",
    "save_sharded_checkpoint": True,
}

print("Text neuron config:")
for k, v in text_neuron_config.items():
    print(f"  {k}: {v}")

Text neuron config:
  batch_size: 1
  ctx_batch_size: 1
  tkg_batch_size: 1
  seq_len: 4096
  max_context_length: 4096
  torch_dtype: bfloat16
  tp_degree: 4
  world_size: 4
  enable_bucketing: True
  context_encoding_buckets: [512, 1024, 2048, 4096]
  token_generation_buckets: [512, 1024, 2048, 4096]
  fused_qkv: True
  qkv_kernel_enabled: False
  mlp_kernel_enabled: False
  attn_kernel_enabled: False
  logical_neuron_cores: 1
  cc_pipeline_tiling_factor: 1
  rpl_reduce_dtype: bfloat16
  attention_dtype: bfloat16
  cast_type: as-declared
  save_sharded_checkpoint: True


In [8]:
# Set artifact directory
os.environ["NEURON_COMPILED_ARTIFACTS"] = ARTIFACT_DIR
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print(f"Artifacts will be saved to: {ARTIFACT_DIR}")
print(f"\nStarting compilation...")
print(f"This will take 15-45 minutes on inf2.24xlarge.")

from vllm import LLM, SamplingParams

compile_start = time.time()

llm = LLM(
    model=BASE_MODEL_DIR,
    tokenizer=BASE_MODEL_DIR,
    trust_remote_code=True,
    dtype="bfloat16",
    tensor_parallel_size=TP_DEGREE,
    max_num_seqs=BATCH_SIZE,
    max_model_len=MAX_MODEL_LEN,
    swap_space=0,
    enable_lora=True,
    max_loras=len(LORA_ADAPTERS),
    max_lora_rank=32,
    additional_config=dict(
        override_neuron_config=dict(
            text_neuron_config=text_neuron_config,
            lora_ckpt_json=lora_json_path,
        )
    ),
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)

compile_time = time.time() - compile_start
print(f"\nCompilation + loading completed in {compile_time:.1f}s ({compile_time/60:.1f} min)")

Artifacts will be saved to: /home/ubuntu/neuron-artifacts/qwen3-14b-tp4-lora

Starting compilation...
This will take 15-45 minutes on inf2.24xlarge.
INFO 03-17 14:17:19 [__init__.py:43] Available plugins for group vllm.platform_plugins:
INFO 03-17 14:17:19 [__init__.py:45] - neuron -> vllm_neuron:register
INFO 03-17 14:17:19 [__init__.py:48] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 03-17 14:17:19 [__init__.py:217] Platform plugin neuron is activated
INFO 03-17 14:17:21 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 03-17 14:17:21 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
INFO 03-17 14:17:22 [utils.py:253] non-default args: {'tokenizer': '/home/ubuntu/Qwen3-14B', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 4096, 'tensor_parallel_size': 4, 'enable_prefix_cachi

[2026-03-17 14:17:22] INFO platform.py:108: Applying Neuron config overrides
[2026-03-17 14:17:22] INFO platform.py:125: Neuron config overrides applied successfully
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
[2026-03-17 14:17:24] INFO platform_overrides.py:22: Skipping attention head divisibility check for Neuron platform
[2026-03-17 14:17:24] INFO platform.py:167: Neuron OpenAI serving overrides applied successfully
[2026-03-17 14:17:24] INFO platform.py:272: The custom Neuron scheduler will disable chunked prefill and schedule requests using the continuous batching mechanism, prioritizing prefill over decode.
[2026-03-17 14:17:24] INFO platform.py:285: Neuron custom scheduler default: max_num_batched_tokens set to 131072. Override with --max-num-batched-tokens if needed.
[2026-03-17 14:17:24] WARNING platform.py:312: Pin memory is not supported on Neuron.


## 4. Verify Inference with LoRA Adapters

Test that the model produces meaningful output with each LoRA adapter.

**Known issue (internal ticket filed):** vLLM-neuron 0.4.1 requires every request to include a
`lora_request` when `enable_lora=True`. Sending a request without one (i.e., base model inference)
crashes with `AttributeError: 'NoneType' object has no attribute 'lora_name'`. This will be resolved
in a future vLLM-neuron release. Until then, always pass a `lora_request` with every generate call.

In [9]:
from vllm.lora.request import LoRARequest

sampling = SamplingParams(top_k=1, max_tokens=128, temperature=0.0)

test_prompts = [
    "Explain quantum computing in one paragraph.",
    "Write a short Python function that checks if a number is prime.",
    "What are the main differences between TCP and UDP?",
]

# NOTE: Base model (no LoRA) inference is not currently supported when enable_lora=True.
# See known issue note above. All requests must include a lora_request.

# Test with LoRA Adapter A
print("=" * 60)
print(f"ADAPTER A: {LORA_ADAPTERS['adapter_a']}")
print("=" * 60)

lora_req_a = LoRARequest("adapter_a", lora_int_id=1, lora_path=" ")

for prompt in test_prompts:
    t0 = time.time()
    outputs = llm.generate([{"prompt": prompt}], sampling, lora_request=[lora_req_a])
    latency = time.time() - t0
    text = outputs[0].outputs[0].text
    n_tokens = len(outputs[0].outputs[0].token_ids)
    print(f"\nPrompt: {prompt}")
    print(f"Tokens: {n_tokens}, Latency: {latency:.2f}s, Throughput: {n_tokens/latency:.1f} tok/s")
    print(f"Output: {text[:300]}")
    print("-" * 40)

ADAPTER A: nicoboss/Qwen3-14B-Uncensored-Lora

Prompt: Explain quantum computing in one paragraph.
Tokens: 128, Latency: 5.13s, Throughput: 25.0 tok/s
Output:  Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. Unlike classical computers, which use bits as their basic unit of information, quantum computers use qubits, which can exist in multiple states at 
----------------------------------------

Prompt: Write a short Python function that checks if a number is prime.
Tokens: 128, Latency: 4.66s, Throughput: 27.4 tok/s
Output:  The function should return True if the number is prime, and False otherwise. The function should handle edge cases such as numbers less than 2, even numbers, and numbers with multiple factors. The function should be efficient and not use any unnecessary loops or calculations. Here's a Python functi
----------------------------------------

Prompt: What are the ma


Adding requests: 100%|██████████| 1/1 [00:00<00:00, 76.77it/s]

Processed prompts: 100%|██████████| 1/1 [00:05<00:00,  5.11s/it, est. speed input: 1.57 toks/s, output: 25.06 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1557.48it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.66s/it, est. speed input: 2.79 toks/s, output: 27.45 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1776.49it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.63s/it, est. speed input: 2.16 toks/s, output: 27.63 toks/s]


In [10]:
# Test with LoRA Adapter B
print("=" * 60)
print(f"ADAPTER B: {LORA_ADAPTERS['adapter_b']}")
print("=" * 60)

lora_req_b = LoRARequest("adapter_b", lora_int_id=1, lora_path=" ")

for prompt in test_prompts:
    t0 = time.time()
    outputs = llm.generate([{"prompt": prompt}], sampling, lora_request=[lora_req_b])
    latency = time.time() - t0
    text = outputs[0].outputs[0].text
    n_tokens = len(outputs[0].outputs[0].token_ids)
    print(f"\nPrompt: {prompt}")
    print(f"Tokens: {n_tokens}, Latency: {latency:.2f}s, Throughput: {n_tokens/latency:.1f} tok/s")
    print(f"Output: {text[:300]}")
    print("-" * 40)

ADAPTER B: Wuhall/Qwen3-14B-LoRA

Prompt: Explain quantum computing in one paragraph.
Tokens: 128, Latency: 4.65s, Throughput: 27.6 tok/s
Output:  Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations and process information. Unlike classical computers, which use bits to represent information as either 0 or 1, quantum computers use qubits, which can exist in multiple states at once, all
----------------------------------------

Prompt: Write a short Python function that checks if a number is prime.
Tokens: 128, Latency: 4.64s, Throughput: 27.6 tok/s
Output:  The function should return True if the number is prime, and False otherwise. The function should handle the case where the number is less than 2 by returning False. The function should also handle the case where the number is 2 by returning True. The function should not use any built-in functions o
----------------------------------------

Prompt: What are the main difference


Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2227.46it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.64s/it, est. speed input: 1.72 toks/s, output: 27.56 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1813.36it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.64s/it, est. speed input: 2.80 toks/s, output: 27.57 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2200.58it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.63s/it, est. speed input: 2.16 toks/s, output: 27.64 toks/s]


## 5. Benchmark Throughput

Run a structured benchmark with multiple prompts to measure steady-state performance.

Only LoRA adapter benchmarks are run (see known issue in Section 4 regarding base model inference).

In [11]:
import statistics

benchmark_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a recursive Fibonacci function in Python with memoization.",
    "What is the difference between a stack and a queue?",
    "Describe the process of photosynthesis.",
    "What are the advantages of microservices architecture?",
    "Explain how a neural network learns through backpropagation.",
    "What is the CAP theorem in distributed systems?",
    "Write a SQL query to find the second highest salary.",
    "Explain the concept of polymorphism in object-oriented programming.",
    "What are the key features of the Rust programming language?",
]

sampling_bench = SamplingParams(top_k=1, max_tokens=256, temperature=0.0)

def run_benchmark(llm, prompts, sampling_params, lora_request, label):
    """Run benchmark with a specific LoRA adapter.
    
    Args:
        lora_request: A LoRARequest object (required -- see known issue in Section 4).
        label: Human-readable label for this benchmark run.
    """
    results = []
    
    # Warmup
    print(f"  Warmup ({label})...")
    llm.generate([{"prompt": prompts[0]}], sampling_params, lora_request=[lora_request])
    
    print(f"  Running {len(prompts)} prompts...")
    for i, prompt in enumerate(prompts):
        t0 = time.time()
        outputs = llm.generate([{"prompt": prompt}], sampling_params, lora_request=[lora_request])
        latency = time.time() - t0
        n_tokens = len(outputs[0].outputs[0].token_ids)
        tok_per_s = n_tokens / latency if latency > 0 else 0
        results.append({
            "prompt_idx": i,
            "n_tokens": n_tokens,
            "latency_s": round(latency, 3),
            "tok_per_s": round(tok_per_s, 1),
        })
    
    throughputs = [r["tok_per_s"] for r in results]
    latencies = [r["latency_s"] for r in results]
    tokens = [r["n_tokens"] for r in results]
    
    summary = {
        "label": label,
        "num_prompts": len(prompts),
        "mean_throughput_tok_s": round(statistics.mean(throughputs), 1),
        "std_throughput_tok_s": round(statistics.stdev(throughputs), 1) if len(throughputs) > 1 else 0,
        "mean_latency_s": round(statistics.mean(latencies), 2),
        "mean_tokens": round(statistics.mean(tokens), 0),
        "total_tokens": sum(tokens),
        "total_time_s": round(sum(latencies), 1),
        "results": results,
    }
    
    print(f"\n  [{label}] Results:")
    print(f"    Mean throughput: {summary['mean_throughput_tok_s']:.1f} +/- {summary['std_throughput_tok_s']:.1f} tok/s")
    print(f"    Mean latency:   {summary['mean_latency_s']:.2f}s")
    print(f"    Mean tokens:    {summary['mean_tokens']:.0f}")
    print(f"    Total tokens:   {summary['total_tokens']}")
    print(f"    Total time:     {summary['total_time_s']:.1f}s")
    
    return summary

print("Starting benchmarks...\n")

Starting benchmarks...



In [12]:
# Benchmark: Adapter A
print("=" * 60)
print(f"BENCHMARK: Adapter A ({LORA_ADAPTERS['adapter_a']})")
print("=" * 60)
lora_req_a = LoRARequest("adapter_a", lora_int_id=1, lora_path=" ")
results_a = run_benchmark(llm, benchmark_prompts, sampling_bench, lora_request=lora_req_a, label="Adapter A")

BENCHMARK: Adapter A (nicoboss/Qwen3-14B-Uncensored-Lora)
  Warmup (Adapter A)...
  Running 10 prompts...

  [Adapter A] Results:
    Mean throughput: 27.6 +/- 0.0 tok/s
    Mean latency:   9.26s
    Mean tokens:    256
    Total tokens:   2560
    Total time:     92.6s



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2058.05it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.26s/it, est. speed input: 1.19 toks/s, output: 27.66 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2055.02it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.28s/it, est. speed input: 1.19 toks/s, output: 27.60 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1769.00it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.25s/it, est. speed input: 1.19 toks/s, output: 27.69 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1917.83it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.24s/it, est. speed input: 1.19 toks/s, output: 27.70 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1899.59it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.25s/it, est. speed input: 0.76 toks/s, output: 27.67 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1828.38it/s]

Processe

In [13]:
# Benchmark: Adapter B
print("=" * 60)
print(f"BENCHMARK: Adapter B ({LORA_ADAPTERS['adapter_b']})")
print("=" * 60)
lora_req_b = LoRARequest("adapter_b", lora_int_id=1, lora_path=" ")
results_b = run_benchmark(llm, benchmark_prompts, sampling_bench, lora_request=lora_req_b, label="Adapter B")

BENCHMARK: Adapter B (Wuhall/Qwen3-14B-LoRA)
  Warmup (Adapter B)...
  Running 10 prompts...

  [Adapter B] Results:
    Mean throughput: 27.5 +/- 0.1 tok/s
    Mean latency:   9.29s
    Mean tokens:    256
    Total tokens:   2560
    Total time:     92.9s



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2204.05it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.27s/it, est. speed input: 1.19 toks/s, output: 27.62 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2138.86it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.29s/it, est. speed input: 1.18 toks/s, output: 27.57 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1947.22it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.29s/it, est. speed input: 1.18 toks/s, output: 27.57 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2061.08it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.28s/it, est. speed input: 1.19 toks/s, output: 27.60 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2319.86it/s]

Processed prompts: 100%|██████████| 1/1 [00:09<00:00,  9.27s/it, est. speed input: 0.75 toks/s, output: 27.61 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2148.72it/s]

Processe

In [14]:
# Summary table
print("\n" + "=" * 70)
print("BENCHMARK SUMMARY: Qwen3-14B on inf2.24xlarge, tp=4, BF16")
print("=" * 70)
print(f"{'Config':<20} {'Throughput (tok/s)':>20} {'Latency (s)':>15} {'Tokens':>10}")
print("-" * 70)

for r in [results_a, results_b]:
    print(f"{r['label']:<20} {r['mean_throughput_tok_s']:>17.1f} +/- {r['std_throughput_tok_s']:<5.1f} {r['mean_latency_s']:>12.2f}s {r['mean_tokens']:>10.0f}")

print("-" * 70)
print(f"\nConfiguration:")
print(f"  Instance: inf2.24xlarge (12 NeuronCores)")
print(f"  TP: {TP_DEGREE}, Batch size: {BATCH_SIZE}")
print(f"  Max model len: {MAX_MODEL_LEN}")
print(f"  Precision: BF16, ISA kernels: all OFF")
print(f"  Compile time: {compile_time:.0f}s")
print(f"\nNote: Base model (no LoRA) benchmarks not available due to known vLLM-neuron issue.")
print(f"See Section 4 for details. Internal ticket filed.")


BENCHMARK SUMMARY: Qwen3-14B on inf2.24xlarge, tp=4, BF16
Config                 Throughput (tok/s)     Latency (s)     Tokens
----------------------------------------------------------------------
Adapter A                         27.6 +/- 0.0           9.26s        256
Adapter B                         27.5 +/- 0.1           9.29s        256
----------------------------------------------------------------------

Configuration:
  Instance: inf2.24xlarge (12 NeuronCores)
  TP: 4, Batch size: 1
  Max model len: 4096
  Precision: BF16, ISA kernels: all OFF
  Compile time: 753s

Note: Base model (no LoRA) benchmarks not available due to known vLLM-neuron issue.
See Section 4 for details. Internal ticket filed.


In [15]:
# Save results to JSON
results_path = os.path.expanduser("~/qwen3_14b_lora_results.json")
all_results = {
    "config": {
        "model": BASE_MODEL_ID,
        "instance": "inf2.24xlarge",
        "tp_degree": TP_DEGREE,
        "batch_size": BATCH_SIZE,
        "max_model_len": MAX_MODEL_LEN,
        "precision": "bfloat16",
        "isa_kernels": "all_off",
        "compile_time_s": compile_time,
        "lora_adapters": LORA_ADAPTERS,
        "note": "base model (no LoRA) cannot be tested -- vLLM-neuron requires lora_request when enable_lora=True",
    },
    "results": {
        "adapter_a": {k: v for k, v in results_a.items() if k != "results"},
        "adapter_b": {k: v for k, v in results_b.items() if k != "results"},
    },
    "detailed_results": {
        "adapter_a": results_a["results"],
        "adapter_b": results_b["results"],
    },
}

with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to {results_path}")

Results saved to /home/ubuntu/qwen3_14b_lora_results.json


## 6. Check and Upload Artifacts

Verify the compiled artifacts and upload to HuggingFace for reuse on fresh instances.

In [16]:
# Check saved artifacts
print(f"Artifact directory: {ARTIFACT_DIR}\n")

total_size = 0
for root, dirs, files in os.walk(ARTIFACT_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        total_size += size
        rel = os.path.relpath(fpath, ARTIFACT_DIR)
        print(f"  {rel:65s} {size / 1024 / 1024:8.1f} MB")

print(f"\n  TOTAL: {total_size / 1024 / 1024 / 1024:.2f} GB")

# Check for sharded weights
has_text_weights = os.path.exists(os.path.join(ARTIFACT_DIR, "text_model", "weights"))
print(f"\n  Pre-sharded weights: {'YES' if has_text_weights else 'NO'}")

Artifact directory: /home/ubuntu/neuron-artifacts/qwen3-14b-tp4-lora

  model.pt                                                             190.9 MB
  neuron_config.json                                                     0.0 MB

  TOTAL: 0.19 GB

  Pre-sharded weights: NO


In [17]:
# Free model memory before upload
del llm
import gc
gc.collect()
print("Model freed from memory.")

Model freed from memory.


In [18]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# Create repo
api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
print(f"Created repo: {HF_REPO}")

# Upload artifacts
print(f"Uploading artifacts from {ARTIFACT_DIR}...")
print(f"This may take several minutes for ~{total_size / 1024 / 1024 / 1024:.1f} GB of data.")

upload_start = time.time()
api.upload_folder(
    folder_path=ARTIFACT_DIR,
    repo_id=HF_REPO,
    path_in_repo="bs1_tp4",
    repo_type="model",
)
upload_time = time.time() - upload_start
print(f"Artifacts uploaded in {upload_time:.0f}s")

# Upload LoRA adapters (so they're available with the artifacts)
print("Uploading LoRA adapters...")
api.upload_folder(
    folder_path=LORA_DIR,
    repo_id=HF_REPO,
    path_in_repo="lora_adapters",
    repo_type="model",
)

# Upload results
api.upload_file(
    path_or_fileobj=results_path,
    path_in_repo="benchmark_results.json",
    repo_id=HF_REPO,
    repo_type="model",
)

print(f"\nAll uploads complete: https://huggingface.co/{HF_REPO}")

Created repo: jburtoft/Qwen3-14B-neuron-inf2-tp4-lora
Uploading artifacts from /home/ubuntu/neuron-artifacts/qwen3-14b-tp4-lora...
This may take several minutes for ~0.2 GB of data.
Artifacts uploaded in 7s
Uploading LoRA adapters...

All uploads complete: https://huggingface.co/jburtoft/Qwen3-14B-neuron-inf2-tp4-lora



Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            


  ...en3-14b-tp4-lora/model.pt:   1%|          | 1.06MB /  200MB            


  ...en3-14b-tp4-lora/model.pt:   1%|          | 1.06MB /  200MB            
Processing Files (0 / 1)      :   1%|          | 1.06MB /  200MB, 1.32MB/s  

New Data Upload               :   2%|▏         | 1.06MB / 67.0MB, 1.32MB/s  


  ...en3-14b-tp4-lora/model.pt:   1%|          | 2.11MB /  200MB            
Processing Files (0 / 1)      :   1%|          | 2.11MB /  200MB, 2.11MB/s  

New Data Upload               :   3%|▎         | 2.11MB / 67.0MB, 2.11MB/s  


  ...en3-14b-tp4-lora/model.pt:   3%|▎         | 5.28MB /  200MB            
Processing Files (0 / 1)      :   3%|▎         | 5.28MB /  200MB, 4.40MB/s  

New Data Upload               :   4%|▍         | 5.28MB /  134MB, 4.40MB/s  


  ...en3-14b-tp4-lora/model.pt:   5%|▍         | 9.50MB /  200MB     

In [19]:
# Upload model card
model_card = f"""---
tags:
- neuron
- aws
- inf2
- qwen3
- lora
- pre-compiled
library_name: neuronx-distributed-inference
---

# Qwen3-14B with LoRA -- Pre-compiled for AWS Inferentia2

Pre-compiled artifacts for running [Qwen/Qwen3-14B](https://huggingface.co/Qwen/Qwen3-14B)
with LoRA adapters on AWS Inferentia2 (inf2.24xlarge).

## Configuration

| Setting | Value |
|---------|-------|
| Instance type | inf2.24xlarge (12 NeuronCores) |
| Tensor parallel | 4 |
| Batch size | 1 |
| Max sequence length | {MAX_MODEL_LEN} |
| Data type | BF16 |
| ISA Kernels | All OFF (required on inf2) |
| Compile time | {compile_time:.0f}s |
| SDK | Neuron SDK 2.28 (DLAMI 20260227) |
| NxD Inference | 0.8.x |
| vLLM-neuron | 0.4.1 |

## Benchmark Results

| Config | Throughput (tok/s) | Latency (s) | Avg Tokens |
|--------|-------------------|-------------|------------|
| Adapter A (nicoboss/Uncensored) | {results_a['mean_throughput_tok_s']:.1f} +/- {results_a['std_throughput_tok_s']:.1f} | {results_a['mean_latency_s']:.2f} | {results_a['mean_tokens']:.0f} |
| Adapter B (Wuhall/LoRA) | {results_b['mean_throughput_tok_s']:.1f} +/- {results_b['std_throughput_tok_s']:.1f} | {results_b['mean_latency_s']:.2f} | {results_b['mean_tokens']:.0f} |

## Included LoRA Adapters

| Adapter | Source | Rank | Alpha | Target Modules |
|---------|--------|------|-------|---------------|
| adapter_a | nicoboss/Qwen3-14B-Uncensored-Lora | 32 | 16 | q/k/v/o/gate/up/down_proj |
| adapter_b | Wuhall/Qwen3-14B-LoRA | 32 | 32 | q/k/v/o/gate/up/down_proj |

## Quick Start on a Fresh inf2.24xlarge

```bash
# 1. Activate Neuron venv
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
pip install huggingface_hub

# 2. Download base model (required -- artifacts don't include base weights)
python -c "
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Qwen/Qwen3-14B', local_dir='Qwen3-14B',
                  ignore_patterns=['*.gguf', '*.md', 'original/*'])
"

# 3. Download pre-compiled artifacts + LoRA adapters
python -c "
from huggingface_hub import snapshot_download
snapshot_download(repo_id='{HF_REPO}', local_dir='neuron-artifacts')
"

# 4. Create LoRA config JSON (update paths to match your layout)
python -c "
import json, os
config = {{
    'lora-ckpt-dir': os.path.abspath('neuron-artifacts/lora_adapters'),
    'lora-ckpt-paths': {{
        'adapter_a': 'adapter_a',
        'adapter_b': 'adapter_b',
    }},
    'lora-ckpt-paths-cpu': {{}}
}}
with open('neuron-artifacts/lora_adapters/adapters.json', 'w') as f:
    json.dump(config, f, indent=2)
print('LoRA config written')
"

# 5. Set environment and run
export NEURON_COMPILED_ARTIFACTS=$PWD/neuron-artifacts/bs1_tp4
export VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
export VLLM_USE_V1=1
```

```python
import os
os.environ["NEURON_COMPILED_ARTIFACTS"] = "neuron-artifacts/bs1_tp4"
os.environ["VLLM_NEURON_FRAMEWORK"] = "neuronx-distributed-inference"
os.environ["VLLM_USE_V1"] = "1"

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model="Qwen3-14B",
    trust_remote_code=True,
    dtype="bfloat16",
    tensor_parallel_size=4,
    max_num_seqs=1,
    max_model_len=4096,
    swap_space=0,
    enable_lora=True,
    max_loras=2,
    max_lora_rank=32,
    additional_config=dict(
        override_neuron_config=dict(
            text_neuron_config={{...}},  # same config as Section 3
            lora_ckpt_json="neuron-artifacts/lora_adapters/adapters.json",
        )
    ),
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)

# NOTE: When enable_lora=True, EVERY request must include a lora_request.
# Sending lora_request=None causes an error (known issue, internal ticket filed).
sampling = SamplingParams(top_k=1, max_tokens=256, temperature=0.0)
lora_req = LoRARequest("adapter_a", lora_int_id=1, lora_path=" ")
outputs = llm.generate(
    [{{"prompt": "Hello! Explain quantum computing briefly."}}],
    sampling,
    lora_request=[lora_req],
)
print(outputs[0].outputs[0].text)
```

## Important Notes

1. **Base model weights required**: Download `Qwen/Qwen3-14B` separately (~30 GB).
2. **LoRA always required**: When `enable_lora=True`, every request MUST include a `lora_request`. Omitting it causes `AttributeError`. Internal ticket filed.
3. **inf2.24xlarge required**: tp=4 needs 4+ NeuronCores. Smaller inf2 instances won't work.
4. **No FP8 on inf2**: BF16 only. Customer's GPU `--quantization fp8` cannot be replicated.
5. **SDK-specific**: Artifacts work with Neuron SDK 2.28 (DLAMI 20260227) only.
6. **Update LoRA paths**: The `lora-ckpt-dir` in `adapters.json` must be an absolute path matching your local layout.

## Customer Migration Notes

The customer's GPU command:
```bash
vllm serve Qwen3-14B --quantization fp8 --enable-lora --max-loras 2 --max-lora-rank 32 \\
  --lora-modules main_adapter=<path1> main_adapter_green=<path2>
```

On inf2.24xlarge becomes:
```bash
export NEURON_COMPILED_ARTIFACTS=neuron-artifacts/bs1_tp4
export VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
# LoRA adapters specified via JSON config (see above)
# No --quantization (BF16 only on inf2)
# tensor_parallel_size=4, max_model_len=4096
```
"""

api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="model",
)

print(f"Model card uploaded to https://huggingface.co/{HF_REPO}")

Model card uploaded to https://huggingface.co/jburtoft/Qwen3-14B-neuron-inf2-tp4-lora


## 7. Summary

### What we accomplished

1. Compiled Qwen3-14B (14.8B params, BF16) on inf2.24xlarge at tp=4
2. Integrated two LoRA adapters (rank=32) via NxDI's LoRA serving
3. Benchmarked both adapters (base model benchmarks not available due to known vLLM-neuron issue)
4. Uploaded pre-compiled artifacts to HuggingFace for instant deployment

### Key findings

- **tp=4 on inf2.24xlarge**: Uses 4 of 12 NeuronCores, leaving 8 cores available for DP
- **LoRA throughput**: ~27.3 tok/s with near-zero overhead between adapters
- **Pre-compilation workflow**: Compile once, deploy anywhere (same SDK + instance type)
- **No FP8 on inf2**: BF16 is the only option; customer's GPU FP8 config cannot be directly replicated
- **Known issue**: Base model inference (no LoRA) not supported when `enable_lora=True` in vLLM-neuron 0.4.1. Internal ticket filed.

### For the customer

The customer's GPU command:
```bash
vllm serve Qwen3-14B --quantization fp8 --enable-lora --max-loras 2 --max-lora-rank 32
```

Becomes on inf2.24xlarge:
```bash
export NEURON_COMPILED_ARTIFACTS=neuron-artifacts/bs1_tp4
export VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
# Launch vLLM with LoRA config (see cells above for full config)
```

In [20]:
print("Done! Notebook execution complete.")
print(f"\nArtifacts available at: https://huggingface.co/{HF_REPO}")
print(f"Benchmark results saved to: {results_path}")

Done! Notebook execution complete.

Artifacts available at: https://huggingface.co/jburtoft/Qwen3-14B-neuron-inf2-tp4-lora
Benchmark results saved to: /home/ubuntu/qwen3_14b_lora_results.json
